In [5]:
import torch
import triton
import triton.language as tl

# ── Configs sized to T4's 64 KB smem ──────────────────────────────────────────
# smem/stage = (BLOCK_M*BLOCK_K + BLOCK_K*BLOCK_N) * 2 bytes (fp16)
# 128×128×32: (128×32 + 32×128)×2 = 16 KB/stage → ×4 stages = 64 KB ✓
# 128×256×32: (128×32 + 32×256)×2 = 24.5 KB/stage → ×2 stages = 49 KB ✓
def get_configs():
    return [
        triton.Config({'BLOCK_M': 128, 'BLOCK_N': 128, 'BLOCK_K': 32, 'GROUP_M': 8}, num_warps=4, num_stages=4),
        triton.Config({'BLOCK_M': 128, 'BLOCK_N': 128, 'BLOCK_K': 64, 'GROUP_M': 8}, num_warps=4, num_stages=3),
        triton.Config({'BLOCK_M': 128, 'BLOCK_N': 256, 'BLOCK_K': 32, 'GROUP_M': 8}, num_warps=8, num_stages=2),
        triton.Config({'BLOCK_M': 256, 'BLOCK_N': 128, 'BLOCK_K': 32, 'GROUP_M': 8}, num_warps=8, num_stages=2),
        triton.Config({'BLOCK_M': 64,  'BLOCK_N': 128, 'BLOCK_K': 32, 'GROUP_M': 8}, num_warps=4, num_stages=4),
        triton.Config({'BLOCK_M': 128, 'BLOCK_N': 64,  'BLOCK_K': 32, 'GROUP_M': 8}, num_warps=4, num_stages=4),
        triton.Config({'BLOCK_M': 64,  'BLOCK_N': 64,  'BLOCK_K': 64, 'GROUP_M': 4}, num_warps=4, num_stages=4),
        triton.Config({'BLOCK_M': 128, 'BLOCK_N': 128, 'BLOCK_K': 32, 'GROUP_M': 8}, num_warps=8, num_stages=4),
    ]


@triton.autotune(configs=get_configs(), key=['M', 'N', 'K'])
@triton.jit
def matmul_kernel_v4(
    A_ptr, B_ptr, C_ptr,
    M, N, K,                         # K is pre-padded → exact multiple of BLOCK_K
    stride_am, stride_ak,
    stride_bk, stride_bn,
    stride_cm, stride_cn,
    BLOCK_M: tl.constexpr,
    BLOCK_N: tl.constexpr,
    BLOCK_K: tl.constexpr,
    GROUP_M: tl.constexpr,
):
    # ── L2-friendly grouped tile ordering ────────────────────────────────────
    pid          = tl.program_id(0)
    num_pid_m    = tl.cdiv(M, BLOCK_M)
    num_pid_n    = tl.cdiv(N, BLOCK_N)
    num_in_grp   = GROUP_M * num_pid_n
    group_id     = pid // num_in_grp
    first_pm     = group_id * GROUP_M
    grp_sz       = min(num_pid_m - first_pm, GROUP_M)
    pid_m        = first_pm + (pid % grp_sz)
    pid_n        = (pid % num_in_grp) // grp_sz

    # ── Row/col offsets — modulo trick eliminates M/N masks on loads ─────────
    rm = (pid_m * BLOCK_M + tl.arange(0, BLOCK_M)) % M
    rn = (pid_n * BLOCK_N + tl.arange(0, BLOCK_N)) % N
    rk = tl.arange(0, BLOCK_K)

    # Base pointers
    A_ptrs = A_ptr + rm[:, None] * stride_am + rk[None, :] * stride_ak
    B_ptrs = B_ptr + rk[:, None] * stride_bk + rn[None, :] * stride_bn

    # ── K-loop — NO masks, NO casts, pure fp16 → Tensor Core ─────────────────
    # K is padded to multiple of BLOCK_K in Python wrapper, so
    # range(K // BLOCK_K) is exact — the compiler emits zero branch overhead.
    acc = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)

    for _ in range(0, K // BLOCK_K):
        a = tl.load(A_ptrs)              # fp16 [BLOCK_M, BLOCK_K]
        b = tl.load(B_ptrs)              # fp16 [BLOCK_K, BLOCK_N]
        acc += tl.dot(a, b)              # fp16×fp16→fp32, hits Tensor Cores
        A_ptrs += BLOCK_K * stride_ak
        B_ptrs += BLOCK_K * stride_bk

    # ── Store — ONE mask, at the boundary only ────────────────────────────────
    cm = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
    cn = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)
    mask = (cm[:, None] < M) & (cn[None, :] < N)
    C_ptrs = C_ptr + cm[:, None] * stride_cm + cn[None, :] * stride_cn
    tl.store(C_ptrs, acc.to(tl.float16), mask=mask)


# ── Wrapper — all prep outside the kernel ─────────────────────────────────────
_MAX_BLOCK_K = 64   # largest BLOCK_K in configs above

def _pad_to(x: torch.Tensor, dim: int, multiple: int) -> torch.Tensor:
    r = x.shape[dim] % multiple
    if r == 0:
        return x
    pad_size = multiple - r
    pad = torch.zeros(
        *x.shape[:dim], pad_size, *x.shape[dim+1:],
        device=x.device, dtype=x.dtype
    )
    return torch.cat([x, pad], dim=dim)

def triton_matmul_v4(a_fp16: torch.Tensor, b_fp16: torch.Tensor) -> torch.Tensor:
    """
    a_fp16, b_fp16: already fp16 and contiguous — cast ONCE outside this call.
    Returns fp16 result.
    """
    M, K = a_fp16.shape
    _,  N = b_fp16.shape

    # Pad K so inner loop is exact — zero pads don't affect result
    a_p = _pad_to(a_fp16, dim=1, multiple=_MAX_BLOCK_K)
    b_p = _pad_to(b_fp16, dim=0, multiple=_MAX_BLOCK_K)
    K_pad = a_p.shape[1]

    c = torch.empty((M, N), device=a_fp16.device, dtype=torch.float16)

    grid = lambda meta: (
        triton.cdiv(M, meta['BLOCK_M']) * triton.cdiv(N, meta['BLOCK_N']),
    )
    matmul_kernel_v4[grid](
        a_p, b_p, c,
        M, N, K_pad,
        a_p.stride(0), a_p.stride(1),
        b_p.stride(0), b_p.stride(1),
        c.stride(0),   c.stride(1),
    )
    return c


# ── Benchmark ─────────────────────────────────────────────────────────────────
def bench(fn, warmup=30, rep=200):
    for _ in range(warmup): fn()
    s = torch.cuda.Event(enable_timing=True)
    e = torch.cuda.Event(enable_timing=True)
    s.record()
    for _ in range(rep): fn()
    e.record()
    torch.cuda.synchronize()
    return s.elapsed_time(e) / rep

def benchmark(M, N, K):
    # All dtype work happens HERE — never inside the timed lambda
    a32 = torch.randn(M, K, device='cuda', dtype=torch.float32)
    b32 = torch.randn(K, N, device='cuda', dtype=torch.float32)
    a16 = a32.to(torch.float16).contiguous()
    b16 = b32.to(torch.float16).contiguous()

    t_ms  = bench(lambda: triton_matmul_v4(a16, b16))
    pt_ms = bench(lambda: torch.matmul(a16, b16))     # apples-to-apples: both fp16

    tflops = lambda ms: 2*M*N*K / (ms*1e-3) / 1e12
    print(f"[{M}x{K}x{N}]  "
          f"Triton: {t_ms:.3f} ms ({tflops(t_ms):.2f} TFLOPS)  |  "
          f"PyTorch(fp16): {pt_ms:.3f} ms ({tflops(pt_ms):.2f} TFLOPS)  |  "
          f"Speedup: {pt_ms/t_ms:.2f}x")

def verify():
    M, K, N = 1024, 1024, 1024
    a = torch.randn(M, K, device='cuda', dtype=torch.float16)
    b = torch.randn(K, N, device='cuda', dtype=torch.float16)
    ref = torch.matmul(a, b)
    out = triton_matmul_v4(a, b)
    err = (ref - out).abs().max().item()
    print(f"Correctness — max|diff|: {err:.5f}  {'PASS ✓' if err < 1.0 else 'FAIL ✗'}")

if __name__ == "__main__":
    print("=== T4 Matmul v4 — FP16 Tensor Core, apples-to-apples ===\n")
    verify()
    print()
    for sz in [(512,512,512),(1024,1024,1024),(2048,2048,2048),
               (4096,4096,4096),(8192,8192,8192)]:
        benchmark(*sz)



=== T4 Matmul v4 — FP16 Tensor Core, apples-to-apples ===

Correctness — max|diff|: 0.06250  PASS ✓

[512x512x512]  Triton: 0.940 ms (0.29 TFLOPS)  |  PyTorch(fp16): 0.021 ms (12.84 TFLOPS)  |  Speedup: 0.02x
[1024x1024x1024]  Triton: 8.367 ms (0.26 TFLOPS)  |  PyTorch(fp16): 0.072 ms (29.93 TFLOPS)  |  Speedup: 0.01x
[2048x2048x2048]  Triton: 67.819 ms (0.25 TFLOPS)  |  PyTorch(fp16): 0.975 ms (17.63 TFLOPS)  |  Speedup: 0.01x
[4096x4096x4096]  Triton: 547.755 ms (0.25 TFLOPS)  |  PyTorch(fp16): 7.832 ms (17.55 TFLOPS)  |  Speedup: 0.01x
[8192x8192x8192]  Triton: 4390.579 ms (0.25 TFLOPS)  |  PyTorch(fp16): 53.848 ms (20.42 TFLOPS)  |  Speedup: 0.01x
